# Classification of Legal Text: experiments
    **Team 9:** Sai Sathvik Gullipalli (PES2UG24CS430) and Priyanka Mallikarjuna Reddy (PES2UG24CS378).

    This notebook reproduces our workflow on the original textacy Supreme Court corpus.
    It trains TF-IDF, LDA, and Doc2Vec classifiers, selects configurations on validation
    macro-F1, and evaluates the frozen models on a held-out test partition.

    **Local:** install `requirements.txt` in the project's virtual environment and select its kernel.
    **Colab:** upload this notebook, run setup, then upload `team9-project.zip` when prompted.
    Use a Python 3.11 or 3.12 CPU runtime. No paid service or GPU is needed.
    A saved run loads its results without evaluating the test set again.

In [ ]:
import os, sys, json, subprocess, zipfile
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

def extract_trusted_zip(path, target):
    target = Path(target).resolve()
    with zipfile.ZipFile(path) as z:
        for member in z.infolist():
            if not (target / member.filename).resolve().is_relative_to(target):
                raise ValueError("Archive contains an unsafe path")
        z.extractall(target)

def find_project():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for candidate in [parent, parent / "legal_text_classification"]:
            if (candidate / "src" / "legal_text").is_dir():
                return candidate.resolve()
    return None

ROOT = find_project()
if ROOT is None and IS_COLAB:
    print("Upload team9-project.zip from the private GitHub release.")
    from google.colab import files
    for filename in files.upload():
        if filename.endswith(".zip"):
            extract_trusted_zip(filename, Path.cwd())
    ROOT = find_project()
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the extracted legal_text_classification project.")
os.chdir(ROOT)
if IS_COLAB:
    # Run this setup cell before importing NumPy/scikit-learn/gensim. If those
    # libraries were previously imported, restart the session after installation.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".build" / "matplotlib"))
print("Project:", ROOT)
print("Python:", sys.version.split()[0])

## Train or load an existing run
    Feature learning uses training opinions only. The subprocess starts with fixed
    hashing and thread settings for reproducibility. Training can take tens of minutes
    on a CPU, depending on hardware. Use the saved demo bundle during the review.

In [ ]:
metrics_path = ROOT / "results" / "metrics.json"
if metrics_path.exists():
    print("Loading completed results. No models are retrained and the test set is not reevaluated.")
else:
    env = dict(os.environ, PYTHONHASHSEED="42", OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
    env["PYTHONPATH"] = str(ROOT / "src")
    subprocess.run([sys.executable, "-m", "legal_text.train", "--project-root", str(ROOT)], env=env, check=True)

## Dataset, cleaning, and fixed partitions

In [ ]:
import pandas as pd
from IPython.display import display, Image
summary = json.loads((ROOT / "results/dataset_summary.json").read_text())
provenance = json.loads((ROOT / "results/provenance.json").read_text())
print("Dataset SHA256:", provenance["sha256"])
print("Source:", provenance["source_page"])
print("Raw records:", summary["raw_records"], "Retained:", summary["retained_records"])
print("Excluded rare categories:", summary["excluded_rare_categories"])
display(pd.DataFrame({n: v["class_counts"] for n, v in summary["splits"].items()}).fillna(0).astype(int))
manifest = pd.read_csv(ROOT / "results/split_manifest.csv", keep_default_na=False)
assert manifest.groupby("group")["split"].nunique().max() == 1
assert manifest.groupby("text_hash")["split"].nunique().max() == 1
print("Verified: case groups and identical texts never cross partitions.")

## Validation selection
Macro-F1 gives each category equal weight. Test scores do not participate in this choice.

In [ ]:
candidates = pd.read_csv(ROOT / "results/validation_candidates.csv")
display(candidates[["model", "topics", "C", "accuracy", "macro_f1", "weighted_f1"]])
selection = json.loads((ROOT / "results/selection.json").read_text())
print("Validation-selected winner:", selection["winner"])
print("Tie-breaks:", selection["selection_metric"])

## Frozen test results

In [ ]:
metrics = json.loads(metrics_path.read_text())
display(pd.DataFrame(metrics).T)
display(Image(filename=str(ROOT / "results/model_comparison.png")))
for name in ("tfidf", "lda", "doc2vec"):
    print(name)
    display(pd.read_csv(ROOT / f"results/{name}_per_category.csv"))
display(Image(filename=str(ROOT / f"results/{selection['winner']}_confusion_matrix.png")))

## Interpretation
    LDA uses count features because its generative model describes word counts.
    This differs from the reference report's stated TF-IDF input. Doc2Vec can discover
    similarity between opinions, but that similarity does not establish legal authority.
    Random grouped splits measure performance on this historical corpus, not generalization
    to future cases. Small categories and possible source metadata mismatches limit conclusions.

    See `docs/viva.md` for explanations and `results/` for all exported evidence.
    Open `02_live_demo.ipynb` for interactive inference without retraining.

In [ ]:
if IS_COLAB:
    import shutil
    archive = shutil.make_archive(str(ROOT / "team9-model-bundle"), "zip", root_dir=ROOT, base_dir="artifacts")
    from google.colab import files
    files.download(archive)
else:
    print("Saved artifacts:", ROOT / "artifacts")